# Composing local operators and using AMG

A local problem exposes its operator, signed trace coupling, source, kernel and physical mean constraints. The global coupling is independent of how these matrices were assembled. CPU AMG acts on a projected and pinned SPD local complement, not directly on the global saddle matrix.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from examples.formulations.application import brinkman as solve_brinkman
from examples.formulations.application import elasticity as solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pymhm import Equation, LocalEquations, MultiscaleProblem, compile_local_equations

local_forms = LocalEquations(
    [[1.0, -1.0], [-1.0, 1.0]],
    [0.0, 0.0],
    np.eye(2),
    -np.eye(2),
    [0, 1],
    kernel=np.ones((2, 1)),
    moments=np.full((2, 1), 0.5),
)
declared = MultiscaleProblem(Equation(0, [0.0, -1.0, 0.0]), lambda item: local_forms, [0], 2, (1,))
system = assemble(declared)
result = system.solve()
np.testing.assert_allclose(result.fields[0], [0.0, 1.0], atol=1e-12)
local = compile_local_equations(local_forms).problem
lu = local.condense("scipy")
amg = local.condense("pyamg")
np.testing.assert_allclose(lu.lifts, amg.lifts, atol=1e-12)
mesh = TriangleMesh.unit_square(2)
solution_definition = define_darcy(mesh, dirichlet=lambda x: 1 + x[:, 0])
solution_system = assemble(solution_definition.problem, solvers=SolverConfig(local_solver="pyamg"))
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(lambda x: 1 + x[:, 0]) < 1e-09
print("AMG MHM pressure error", solution.l2_error(lambda x: 1 + x[:, 0]))


GPU AMG uses the optional AmgX adapter. CPU/GPU backends must meet the same residual criterion, and transfer/setup costs belong in timing. No novelty claim about the first use of AMG with MHM is made.